# Wild Apple Sampling Monthly Diagnostics Debug

Debug notebook focused only on the pipeline up to sampling.

Scope kept intentionally narrow:
- initialize local GEE Python workflow
- build ROI and monthly S2/S1 feature stacks
- construct wild apple, rectangle, and WorldCover sample regions
- print per-component sampling counts
- print month-by-month S2 and S1 sampling counts

Deferred on purpose in this debug copy:
- local RF training
- feature selection
- cubic fitting
- SHAP
- map post-processing and area statistics


In [ ]:
from pathlib import Path
import warnings

import ee
import geemap
import folium
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score, classification_report, cohen_kappa_score, confusion_matrix, f1_score
from sklearn.model_selection import GroupShuffleSplit

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")

OUTPUT_DIR = Path("outputs") / "wildapple_localrun_2021"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

CONFIG = {
    "gee_project": "kindle-400911",
    "year": 2021,
    "start_month": 3,
    "end_month": 10,
    "scale": 20,
    "random_seed": 42,
    "wildapple_asset": "projects/kindle-400911/assets/wildapple_sample",
    "worldcover_points_per_class": 150,
    "enable_wildapple_buffer": True,
    "wildapple_buffer_m": 20,
    "rect_sample_scale": 30,
    "wildapple_rect_group_id": "wildapple_rect_01",
    "wildapple_rect_corners": [
        [82.77484146, 43.20874077],
        [82.77116641, 43.2112944],
        [82.77223955, 43.21193612],
        [82.7756369, 43.20940169],
    ],
    "adm0_name": "China",
    "adm1_name": "Xinjiang Uygur Zizhiqu",
    "adm2_name": "Ili Kazakh",
    "valley_elevation_threshold_m": 1800,
    "valley_slope_threshold_deg": 12,
    "postprocess_min_patch_pixels": 150,
}

CLASS_INFO = {
    1: "Wild Apple Forest",
    2: "Other Forest",
    3: "Cropland",
    4: "Grassland/Shrub",
    5: "Urban/Bare",
    6: "Water/Snow/Ice",
}

MONTHS = list(range(CONFIG["start_month"], CONFIG["end_month"] + 1))
PROJ = None
OPTICAL_SERIES_PREFIXES = ["NDVI", "EVI", "NDMI", "LSWI"]
SAR_SERIES_PREFIXES = ["VV", "VH", "VV_VH_ratio", "VV_db", "VH_db"]
SUMMARY_PREFIXES = OPTICAL_SERIES_PREFIXES + SAR_SERIES_PREFIXES

print(f"Output directory: {OUTPUT_DIR.resolve()}")
print(f"Months: {MONTHS}")
        


In [ ]:
ee.Initialize(project=CONFIG["gee_project"])
PROJ = ee.Projection("EPSG:3857").atScale(CONFIG["scale"])
print("Earth Engine initialized.")
        


## 1. ROI and core utilities


In [ ]:
def get_ili_roi():
    admin = (
        ee.FeatureCollection("FAO/GAUL/2015/level2")
        .filter(ee.Filter.eq("ADM0_NAME", CONFIG["adm0_name"]))
        .filter(ee.Filter.eq("ADM1_NAME", CONFIG["adm1_name"]))
        .filter(ee.Filter.eq("ADM2_NAME", CONFIG["adm2_name"]))
    )
    return admin.geometry()


def month_tag(month):
    return f"M{int(month):02d}"


def month_band_names(prefix):
    return [f"{prefix}_{month_tag(month)}" for month in MONTHS]


def add_string_id(fc, id_field="sample_id", prefix="sample"):
    fc = ee.FeatureCollection(fc)
    size = fc.size()

    def _non_empty(collection):
        collection = ee.FeatureCollection(collection)
        collection_list = collection.toList(collection.size())

        def _mapper(i):
            i = ee.Number(i)
            feature = ee.Feature(collection_list.get(i))
            sample_id = ee.String(prefix).cat("_").cat(i.format("%05d"))
            return feature.set(id_field, sample_id)

        return ee.FeatureCollection(ee.List.sequence(0, collection.size().subtract(1)).map(_mapper))

    return ee.FeatureCollection(ee.Algorithms.If(size.gt(0), _non_empty(fc), fc))


def build_rect_geometry(corners):
    return ee.Geometry.Polygon([corners], None, False)


ROI = get_ili_roi()
print("ROI ready.")

roi_center = ROI.centroid(100).coordinates().getInfo()
roi_geojson = ee.Feature(ROI).getInfo()

Map = folium.Map(location=[roi_center[1], roi_center[0]], zoom_start=7, control_scale=True)
folium.TileLayer(
    tiles="https://mt1.google.com/vt/lyrs=s&x={x}&y={y}&z={z}",
    attr="Google",
    name="Google Satellite",
    overlay=False,
    control=True,
).add_to(Map)
folium.GeoJson(
    data=roi_geojson,
    name="ROI",
    style_function=lambda _: {"color": "#ffcc00", "weight": 2, "fillOpacity": 0.0},
).add_to(Map)
folium.LayerControl().add_to(Map)
Map
        


## 2. Sentinel-2, Sentinel-1, terrain, and monthly feature image


In [ ]:
S2_RAW_BANDS = ["B2", "B3", "B4", "B8", "B11", "B12"]
S2_RENAMED_BANDS = ["blue", "green", "red", "nir", "swir1", "swir2"]
S2_QA_COLLECTION = ee.ImageCollection("GOOGLE/CLOUD_SCORE_PLUS/V1/S2_HARMONIZED")
S2_QA_BAND = "cs_cdf"
S2_CLEAR_THRESHOLD = 0.60


def add_optical_indices(image):
    ndvi = image.normalizedDifference(["nir", "red"]).rename("NDVI")
    evi = image.expression(
        "2.5 * ((NIR - RED) / (NIR + 6 * RED - 7.5 * BLUE + 1))",
        {"NIR": image.select("nir"), "RED": image.select("red"), "BLUE": image.select("blue")},
    ).rename("EVI")
    ndmi = image.normalizedDifference(["nir", "swir1"]).rename("NDMI")
    lswi = image.normalizedDifference(["nir", "swir2"]).rename("LSWI")
    ndbi = image.normalizedDifference(["swir1", "nir"]).rename("NDBI")
    bsi = image.expression(
        "((SWIR + RED) - (NIR + BLUE)) / ((SWIR + RED) + (NIR + BLUE))",
        {"SWIR": image.select("swir1"), "RED": image.select("red"), "NIR": image.select("nir"), "BLUE": image.select("blue")},
    ).rename("BSI")
    gcvi = image.expression("(NIR / GREEN) - 1", {"NIR": image.select("nir"), "GREEN": image.select("green")}).rename("GCVI")
    return image.addBands([ndvi, evi, ndmi, lswi, ndbi, bsi, gcvi])


def preprocess_s2(image):
    image = image.updateMask(image.select(S2_QA_BAND).gte(S2_CLEAR_THRESHOLD))
    image = image.select(S2_RAW_BANDS).rename(S2_RENAMED_BANDS).multiply(0.0001)
    return add_optical_indices(image).resample("bilinear").reproject(PROJ)


def load_monthly_s2(month):
    start_date = ee.Date.fromYMD(CONFIG["year"], month, 1)
    end_date = start_date.advance(1, "month")
    collection = (
        ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
        .filterBounds(ROI)
        .filterDate(start_date, end_date)
        .linkCollection(S2_QA_COLLECTION, [S2_QA_BAND])
        .map(preprocess_s2)
    )
    return collection.median().setDefaultProjection(PROJ).clip(ROI)


def preprocess_s1(image):
    vv = image.select("VV")
    vh = image.select("VH")
    vv_db = ee.Image.constant(10).multiply(vv.log10()).rename("VV_db")
    vh_db = ee.Image.constant(10).multiply(vh.log10()).rename("VH_db")
    ratio = vv.divide(vh.max(ee.Image.constant(0.0001))).rename("VV_VH_ratio")
    return image.select(["VV", "VH"]).addBands([vv_db, vh_db, ratio]).resample("bilinear").reproject(PROJ)


def load_monthly_s1(month):
    start_date = ee.Date.fromYMD(CONFIG["year"], month, 1)
    end_date = start_date.advance(1, "month")
    collection = (
        ee.ImageCollection("COPERNICUS/S1_GRD")
        .filterBounds(ROI)
        .filterDate(start_date, end_date)
        .filter(ee.Filter.eq("instrumentMode", "IW"))
        .filter(ee.Filter.listContains("transmitterReceiverPolarisation", "VV"))
        .filter(ee.Filter.listContains("transmitterReceiverPolarisation", "VH"))
        .map(preprocess_s1)
    )
    return collection.median().setDefaultProjection(PROJ).clip(ROI)


def rename_with_month(image, month):
    suffix = month_tag(month)
    new_names = image.bandNames().map(lambda name: ee.String(name).cat("_").cat(suffix))
    return image.rename(new_names)


def build_monthly_feature_image():
    monthly_images = []
    for month in MONTHS:
        optical = load_monthly_s2(month)
        sar = load_monthly_s1(month)
        monthly_images.append(rename_with_month(ee.Image.cat([optical, sar]), month))
    return ee.Image.cat(monthly_images).clip(ROI)


def summary_from_month_stack(image, prefix):
    names = month_band_names(prefix)
    stack = image.select(names)
    stack_mean = stack.reduce(ee.Reducer.mean()).rename(f"{prefix}_mean")
    stack_std = stack.reduce(ee.Reducer.stdDev()).rename(f"{prefix}_std")
    stack_min = stack.reduce(ee.Reducer.min()).rename(f"{prefix}_min")
    stack_max = stack.reduce(ee.Reducer.max()).rename(f"{prefix}_max")
    stack_amp = stack_max.subtract(stack_min).rename(f"{prefix}_amp")

    array_image = stack.toArray()
    peak_index = array_image.arrayArgmax().arrayGet([0])
    peak_month = ee.Image(peak_index).add(CONFIG["start_month"]).rename(f"{prefix}_peak_month")

    first_band = stack.select([0]).rename(f"{prefix}_spring_ref")
    mid_band = stack.select([int(len(MONTHS) / 2)]).rename(f"{prefix}_summer_ref")
    last_band = stack.select([len(MONTHS) - 1]).rename(f"{prefix}_autumn_ref")
    spring_rise = mid_band.subtract(first_band).rename(f"{prefix}_spring_rise")
    autumn_decline = mid_band.subtract(last_band).rename(f"{prefix}_autumn_decline")

    return ee.Image.cat([stack_mean, stack_std, stack_min, stack_max, stack_amp, peak_month, spring_rise, autumn_decline])


def build_summary_image(image):
    return ee.Image.cat([summary_from_month_stack(image, prefix) for prefix in SUMMARY_PREFIXES])


def build_terrain_image():
    dem = ee.Image("USGS/SRTMGL1_003").clip(ROI)
    terrain = ee.Algorithms.Terrain(dem)
    elevation = terrain.select("elevation").rename("elevation")
    slope = terrain.select("slope").rename("slope")
    aspect = terrain.select("aspect").rename("aspect")
    valley_proxy = elevation.lt(CONFIG["valley_elevation_threshold_m"]).And(slope.lt(CONFIG["valley_slope_threshold_deg"]))
    valley_distance = valley_proxy.fastDistanceTransform(128, "pixels", "squared_euclidean").sqrt().multiply(CONFIG["scale"]).rename("dist_to_valley_proxy_m")
    return ee.Image.cat([elevation, slope, aspect, valley_distance])


def build_texture_image(monthly_image):
    ndvi_name = "NDVI_M07" if 7 in MONTHS else f"NDVI_{month_tag(MONTHS[len(MONTHS) // 2])}"
    gray = monthly_image.select(ndvi_name).unitScale(-0.2, 0.8).multiply(100).toInt()
    glcm = gray.glcmTexture(size=3)
    selected = glcm.select([
        f"{ndvi_name}_contrast",
        f"{ndvi_name}_diss",
        f"{ndvi_name}_ent",
        f"{ndvi_name}_idm",
        f"{ndvi_name}_asm",
        f"{ndvi_name}_var",
    ])
    return selected.rename(["NDVI_tex_contrast", "NDVI_tex_diss", "NDVI_tex_ent", "NDVI_tex_idm", "NDVI_tex_asm", "NDVI_tex_var"])


MONTHLY_IMAGE = build_monthly_feature_image()
SUMMARY_IMAGE = build_summary_image(MONTHLY_IMAGE)
TERRAIN_IMAGE = build_terrain_image()
TEXTURE_IMAGE = build_texture_image(MONTHLY_IMAGE)
FEATURE_IMAGE = ee.Image.cat([MONTHLY_IMAGE, SUMMARY_IMAGE, TERRAIN_IMAGE, TEXTURE_IMAGE]).clip(ROI)

print("Feature image band count:", FEATURE_IMAGE.bandNames().size().getInfo())
print("First 20 bands:", FEATURE_IMAGE.bandNames().getInfo()[:20])
        


## 3. Sample loading, optional wild apple buffer expansion, and GEE sampling


In [ ]:
def load_wildapple_samples():
    raw_fc = ee.FeatureCollection(CONFIG["wildapple_asset"])

    def _to_point(feature):
        feature = ee.Feature(feature)
        lon = ee.Number.parse(feature.get("Longitude"))
        lat = ee.Number.parse(feature.get("Latitude"))
        point = ee.Geometry.Point([lon, lat])
        return ee.Feature(point, feature.toDictionary())

    fc = raw_fc.map(_to_point).filterBounds(ROI)
    fc = add_string_id(fc, id_field="sample_id", prefix="wildapple")
    return fc.map(lambda f: ee.Feature(f).set({"class": 1, "source": "wildapple", "group_id": f.get("sample_id")}))


def build_rect_wildapple_samples():
    rect = build_rect_geometry(CONFIG["wildapple_rect_corners"])
    rect_fc = ee.Image.pixelLonLat().sample(
        region=rect,
        scale=CONFIG["rect_sample_scale"],
        geometries=True,
    )
    rect_fc = add_string_id(rect_fc, id_field="sample_id", prefix="wildapple_rect")
    return rect_fc.map(
        lambda f: ee.Feature(f).set({
            "class": 1,
            "source": "wildapple_rect",
            "group_id": CONFIG["wildapple_rect_group_id"],
        })
    )


def build_worldcover_samples():
    worldcover = ee.ImageCollection("ESA/WorldCover/v200")
    wc_2021 = ee.Image(
        ee.Algorithms.If(
            worldcover.filter(ee.Filter.eq("YEAR", 2021)).size().gt(0),
            worldcover.filter(ee.Filter.eq("YEAR", 2021)).first(),
            ee.Algorithms.If(
                worldcover.filter(ee.Filter.eq("year", 2021)).size().gt(0),
                worldcover.filter(ee.Filter.eq("year", 2021)).first(),
                worldcover.first(),
            ),
        )
    ).select("Map")

    wc_to_class = wc_2021.remap([10, 20, 30, 40, 50, 60, 70, 80, 90], [2, 4, 4, 3, 5, 5, 6, 6, 4]).rename("class").clip(ROI)
    fc = wc_to_class.stratifiedSample(
        numPoints=CONFIG["worldcover_points_per_class"] * 5,
        classBand="class",
        region=ROI,
        scale=CONFIG["scale"],
        tileScale=4,
        geometries=True,
        classValues=[2, 3, 4, 5, 6],
        classPoints=[CONFIG["worldcover_points_per_class"]] * 5,
    )
    fc = add_string_id(fc, id_field="sample_id", prefix="worldcover")
    return fc.map(lambda f: ee.Feature(f).set({"source": "worldcover", "group_id": f.get("sample_id")}))


def maybe_expand_wildapple_regions(fc):
    if not CONFIG["enable_wildapple_buffer"]:
        return fc
    return fc.map(lambda f: ee.Feature(f.geometry().buffer(CONFIG["wildapple_buffer_m"]), f.toDictionary()))


def build_training_sample_fc(feature_image):
    wildapple_points = load_wildapple_samples().merge(build_rect_wildapple_samples())
    wildapple_regions = maybe_expand_wildapple_regions(wildapple_points)
    worldcover_points = build_worldcover_samples()
    sampling_regions = worldcover_points.merge(wildapple_regions)
    return feature_image.sampleRegions(
        collection=sampling_regions,
        properties=["class", "source", "sample_id", "group_id"],
        scale=CONFIG["scale"],
        tileScale=4,
        geometries=True,
    )


def diagnose_sampling_count(image, sampling_regions, label):
    sampled = image.sampleRegions(
        collection=sampling_regions,
        properties=["class", "source", "sample_id", "group_id"],
        scale=CONFIG["scale"],
        tileScale=4,
        geometries=False,
    )
    print(f"{label} sample count:", sampled.size().getInfo())
    return sampled


def diagnose_monthly_component_counts(sampling_regions):
    for month in MONTHS:
        tag = month_tag(month)
        s1_month = rename_with_month(load_monthly_s1(month), month)
        # S2 per-month sample counts are intentionally muted in this debug notebook.
        # Current debug focus is fully on S1 coverage collapse.
        diagnose_sampling_count(s1_month, sampling_regions, f"S1_{tag}")


WILDAPPLE_SAMPLES = load_wildapple_samples()
RECT_SAMPLES = build_rect_wildapple_samples()
WORLDCOVER_SAMPLES = build_worldcover_samples()
print("Wild apple sample count:", WILDAPPLE_SAMPLES.size().getInfo())
print("Rectangle sample count:", RECT_SAMPLES.size().getInfo())
print("WorldCover sample count:", WORLDCOVER_SAMPLES.size().getInfo())

SAMPLING_REGIONS = WORLDCOVER_SAMPLES.merge(maybe_expand_wildapple_regions(WILDAPPLE_SAMPLES.merge(RECT_SAMPLES)))
print("Sampling region count:", SAMPLING_REGIONS.size().getInfo())
diagnose_monthly_component_counts(SAMPLING_REGIONS)
diagnose_sampling_count(MONTHLY_IMAGE, SAMPLING_REGIONS, "MONTHLY_IMAGE")
diagnose_sampling_count(SUMMARY_IMAGE, SAMPLING_REGIONS, "SUMMARY_IMAGE")
diagnose_sampling_count(TEXTURE_IMAGE, SAMPLING_REGIONS, "TEXTURE_IMAGE")
diagnose_sampling_count(TERRAIN_IMAGE, SAMPLING_REGIONS, "TERRAIN_IMAGE")

SAMPLE_FC = build_training_sample_fc(FEATURE_IMAGE)
print("Sample feature count:", SAMPLE_FC.size().getInfo())
        


## Debug Exit Point

Stop after this notebook identifies which monthly S2/S1 slices collapse the sample count.
Once sampling is stable, return to `wildapple_localrun_python.ipynb` for the full workflow.
